# Lending Club Data Wrangling Pipeline

This notebook loads the raw Lending Club dataset and performs end-to-end data preparation.

It provides:
- Data cleaning (handling missing values, inconsistent formats, and duplicates)
- Target variable construction using a fixed prediction horizon
- Feature engineering based on borrower financial and credit characteristics
- Separation of fundamental and full feature sets
- Time-based train, validation, and test splits
- Export of clean, analysis-ready datasets and preprocessing artifacts for modeling

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from typing import List, Optional

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
import joblib

RANDOM_STATE = 42

# Update this path to your local file if needed
DATA_PATH = Path("/Users/alex._choo/Desktop/CAP5771/Loan DS/accepted_2007_to_2018Q4.csv/accepted_2007_to_2018Q4.csv")

SNAPSHOT_DATE = pd.Timestamp("2018-12-31")
HORIZON_MONTHS = 12  # set to 24 for 24-month target

RAW_COLS = [
    "id", "loan_status",
    "annual_inc", "emp_length", "home_ownership", "verification_status",
    "fico_range_low", "fico_range_high", "earliest_cr_line",
    "open_acc", "total_acc", "delinq_2yrs", "pub_rec",
    "dti", "revol_bal", "revol_util",
    "loan_amnt", "term", "int_rate", "installment",
    "purpose", "grade", "sub_grade",
    "issue_d", "last_pymnt_d", "addr_state"
]

NON_DEFAULT_FINAL_STATUSES = {
    "Fully Paid",
    "Does not meet the credit policy. Status: Fully Paid",
}

DEFAULT_STATUSES = {
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status: Charged Off",
}

ACTIVE_NONDEFAULT_STATUSES = {
    "Current",
    "In Grace Period",
    "Late (16-30 days)",
    "Late (31-120 days)",
}

VALID_STATUSES = NON_DEFAULT_FINAL_STATUSES | DEFAULT_STATUSES | ACTIVE_NONDEFAULT_STATUSES

# Raw-stage drops only (for fundamental feature set)
FUNDAMENTAL_DROP = ["grade", "sub_grade", "int_rate", "installment"]

_MISSING_TOKENS = {"", "na", "n/a", "none", "null", "nan"}
_DATE_PATTERNS = {
    "%b-%Y": r"^[A-Za-z]{3}-\d{4}$",
    "%Y-%m-%d": r"^\d{4}-\d{2}-\d{2}$",
    "%m/%d/%Y": r"^\d{1,2}/\d{1,2}/\d{4}$",
}

if not DATA_PATH.exists():
    raise FileNotFoundError(f"DATA_PATH not found: {DATA_PATH}")

In [2]:
# Strict helper functions for parsing numeric and datetime columns with better error messages and handling of missing values.
def _normalize_missing_str(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    low = s.str.lower()
    is_missing = s.isna() | low.isin(_MISSING_TOKENS)
    return s.mask(is_missing, pd.NA)


def _to_numeric_strict(s: pd.Series, col_name: str = "column") -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        return pd.to_numeric(s)

    s_clean = _normalize_missing_str(s).str.replace(",", "", regex=False)
    out = pd.Series(np.nan, index=s.index, dtype="float64")
    mask = s_clean.notna()
    if mask.any():
        try:
            out.loc[mask] = pd.to_numeric(s_clean.loc[mask])
        except Exception as exc:
            bad = s_clean.loc[mask].dropna().astype(str).unique()[:10].tolist()
            raise ValueError(f"{col_name}: invalid numeric values. Examples: {bad}") from exc
    return out


def _to_datetime_multi_strict(
    s: pd.Series,
    formats: List[str],
    col_name: str = "date_col"
) -> pd.Series:
    if pd.api.types.is_datetime64_any_dtype(s):
        return s

    s_clean = _normalize_missing_str(s)
    out = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    remaining = s_clean.notna()
    for fmt in formats:
        if fmt not in _DATE_PATTERNS:
            raise ValueError(f"{col_name}: unsupported format pattern key {fmt}")

        pattern = _DATE_PATTERNS[fmt]
        mask_fmt = remaining & s_clean.str.match(pattern, na=False)

        if mask_fmt.any():
            try:
                out.loc[mask_fmt] = pd.to_datetime(s_clean.loc[mask_fmt], format=fmt)
            except Exception as exc:
                bad = s_clean.loc[mask_fmt].dropna().astype(str).unique()[:10].tolist()
                raise ValueError(f"{col_name}: parse failed for format {fmt}. Examples: {bad}") from exc

            remaining = s_clean.notna() & out.isna()

    if remaining.any():
        bad = s_clean.loc[remaining].dropna().astype(str).unique()[:10].tolist()
        raise ValueError(f"{col_name}: unrecognized date format. Examples: {bad}")

    return out

In [3]:
# Load data + explicit status filtering + fixed-horizon target definition with strict date parsing and handling of missing/ambiguous cases.
chunk_iter = pd.read_csv(
    DATA_PATH,
    usecols=RAW_COLS,
    chunksize=200_000,
    low_memory=False
)

chunks = []
for chunk in chunk_iter:
    chunk.columns = chunk.columns.str.strip()

    if "id" in chunk.columns:
        chunk = chunk.rename(columns={"id": "loan_id"})

    # Normalize status strings before filtering
    chunk["loan_status"] = (
        chunk["loan_status"]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.replace(r"Status:\s*", "Status: ", regex=True)
    )

    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
df = df.drop_duplicates(subset=["loan_id"]).reset_index(drop=True)

# Keep only explicit statuses we support
df = df[df["loan_status"].isin(VALID_STATUSES)].copy()

# Strict date parsing
df["issue_d"] = _to_datetime_multi_strict(df["issue_d"], formats=["%b-%Y"], col_name="issue_d")
df["last_pymnt_d"] = _to_datetime_multi_strict(
    df["last_pymnt_d"],
    formats=["%b-%Y", "%Y-%m-%d", "%m/%d/%Y"],
    col_name="last_pymnt_d"
)

# Horizon observability
df["horizon_end"] = df["issue_d"] + pd.DateOffset(months=HORIZON_MONTHS)
eligible = df["horizon_end"] <= SNAPSHOT_DATE

# Event definition (proxy): default status and last payment date within horizon
is_default_status = df["loan_status"].isin(DEFAULT_STATUSES)
is_nondefault_status = df["loan_status"].isin(NON_DEFAULT_FINAL_STATUSES | ACTIVE_NONDEFAULT_STATUSES)

default_in_h = is_default_status & df["last_pymnt_d"].notna() & (df["last_pymnt_d"] <= df["horizon_end"])
default_after_h = is_default_status & df["last_pymnt_d"].notna() & (df["last_pymnt_d"] > df["horizon_end"])

# If status says default but last_pymnt_d missing, event timing uncertain -> drop
uncertain_default = is_default_status & df["last_pymnt_d"].isna()

keep_mask = eligible & (is_nondefault_status | default_in_h | default_after_h) & ~uncertain_default

df_clean = df.loc[keep_mask].copy()
df_clean["target"] = default_in_h.loc[df_clean.index].astype("int8")

assert set(df_clean["target"].unique()) <= {0, 1}

print("Rows after filtering:", len(df_clean))
print("Target distribution:\n", df_clean["target"].value_counts(normalize=True))
print("Status counts:\n", df_clean["loan_status"].value_counts().head(15))

Rows after filtering: 1763639
Target distribution:
 target
0    0.939957
1    0.060043
Name: proportion, dtype: float64
Status counts:
 loan_status
Fully Paid                                              1029307
Current                                                  451136
Charged Off                                              257917
Late (31-120 days)                                        14246
In Grace Period                                            5517
Late (16-30 days)                                          2746
Does not meet the credit policy. Status: Fully Paid        1988
Does not meet the credit policy. Status: Charged Off        749
Default                                                      33
Name: count, dtype: Int64


In [4]:
# Cleaning transformers - these can be used in a pipeline and will be applied to train/val/test splits separately to avoid data leakage. They also include strict parsing and error handling for better robustness.
class ParseDatesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, date_cols: List[str]):
        self.date_cols = date_cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.date_cols:
            if c in X.columns:
                if c in {"issue_d", "earliest_cr_line"}:
                    X[c] = _to_datetime_multi_strict(X[c], ["%b-%Y"], col_name=c)
                elif c == "last_pymnt_d":
                    X[c] = _to_datetime_multi_strict(X[c], ["%b-%Y", "%Y-%m-%d", "%m/%d/%Y"], col_name=c)
        return X


class DateFeaturesTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, date_col="issue_d"):
        self.date_col = date_col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if self.date_col in X.columns:
            X[self.date_col] = _to_datetime_multi_strict(X[self.date_col], ["%b-%Y"], col_name=self.date_col)
            X[self.date_col + "_month"] = X[self.date_col].dt.month
            X[self.date_col + "_weekday"] = X[self.date_col].dt.weekday
            X[self.date_col + "_quarter"] = X[self.date_col].dt.quarter
            X[self.date_col + "_year"] = X[self.date_col].dt.year
            X[self.date_col + "_weekofyear"] = X[self.date_col].dt.isocalendar().week.astype("float64")
        return X


class PercentToFloatTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, cols: List[str]):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.cols:
            if c in X.columns:
                s = (
                    _normalize_missing_str(X[c])
                    .str.replace("%", "", regex=False)
                    .str.replace(",", "", regex=False)
                    .str.strip()
                )
                X[c] = _to_numeric_strict(s, col_name=c)
        return X


class TermExtractor(BaseEstimator, TransformerMixin):
    def __init__(self, col="term"):
        self.col = col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if self.col in X.columns:
            s = _normalize_missing_str(X[self.col]).str.extract(r"(\d+)")[0]
            X[self.col] = _to_numeric_strict(s, col_name=self.col)
        return X


class EmpLengthTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, col="emp_length"):
        self.col = col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        if self.col in X.columns:
            s = _normalize_missing_str(X[self.col]).str.lower()
            s = s.str.replace(r"10\+", "10", regex=True)
            s = s.str.replace(r"<\s*1", "0", regex=True)
            nums = s.str.extract(r"(\d+)", expand=False)
            X[self.col + "_years"] = _to_numeric_strict(nums, col_name=self.col + "_years")
        return X


class FicoAndCreditAgeTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        if "fico_range_low" in X.columns and "fico_range_high" in X.columns:
            X["fico_range_low"] = _to_numeric_strict(X["fico_range_low"], col_name="fico_range_low")
            X["fico_range_high"] = _to_numeric_strict(X["fico_range_high"], col_name="fico_range_high")
            X["fico"] = (X["fico_range_low"] + X["fico_range_high"]) / 2

        if "issue_d" in X.columns and "earliest_cr_line" in X.columns:
            X["issue_d"] = _to_datetime_multi_strict(X["issue_d"], ["%b-%Y"], col_name="issue_d")
            X["earliest_cr_line"] = _to_datetime_multi_strict(X["earliest_cr_line"], ["%b-%Y"], col_name="earliest_cr_line")
            credit_age = (X["issue_d"] - X["earliest_cr_line"]).dt.days / 365.25
            X["credit_age_years"] = credit_age.where(credit_age >= 0, np.nan)

        return X


class CategoricalCleaner(BaseEstimator, TransformerMixin):
    def __init__(self, cols: List[str], rare_thresh=0.01):
        self.cols = cols
        self.rare_thresh = rare_thresh
        self.keep_categories_ = {}

    def fit(self, X, y=None):
        for col in self.cols:
            if col in X.columns:
                s = _normalize_missing_str(X[col]).str.lower()
                vc = s.value_counts(normalize=True, dropna=True)
                top = vc[vc >= self.rare_thresh].index.tolist()
                self.keep_categories_[col] = set(top)
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cols:
            if col in X.columns:
                s = _normalize_missing_str(X[col]).str.lower()
                allowed = self.keep_categories_.get(col, None)
                if allowed is not None:
                    s = s.where(s.isin(allowed) | s.isna(), other="__other__")
                X[col] = s
        return X


class DropColumnsTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, cols: List[str]):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        existing = [c for c in self.cols if c in X.columns]
        return X.drop(columns=existing)


class DerivedFeaturesTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        for c in [
            "revol_bal", "loan_amnt", "installment", "annual_inc",
            "open_acc", "total_acc", "revol_util", "fico"
        ]:
            if c in X.columns:
                X[c] = _to_numeric_strict(X[c], col_name=c)

        # Keep tail signal for revol_util; do not hard clip to 100
        if "revol_util" in X.columns:
            ru = X["revol_util"]
            X["revol_util_gt_100"] = np.where(ru.notna(), (ru > 100).astype(float), np.nan)

        if "revol_bal" in X.columns and "loan_amnt" in X.columns:
            X["revol_to_loan"] = X["revol_bal"] / X["loan_amnt"].replace({0: np.nan})

        if "installment" in X.columns and "annual_inc" in X.columns:
            X["installment_to_monthly_income"] = X["installment"] / (X["annual_inc"] / 12).replace({0: np.nan})

        if "open_acc" in X.columns and "total_acc" in X.columns:
            X["open_to_total_ratio"] = X["open_acc"] / X["total_acc"].replace({0: np.nan})

        if "credit_age_years" in X.columns:
            X["credit_age_bucket"] = pd.cut(
                X["credit_age_years"],
                bins=[-1, 1, 3, 5, 10, 20, 100],
                labels=False
            )

        if "fico" in X.columns and "revol_util" in X.columns:
            X["fico_times_revolutil"] = X["fico"] * (X["revol_util"] / 100.0)

        return X


class LogTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, cols: List[str]):
        self.cols = cols

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.cols:
            if c in X.columns:
                vals = _to_numeric_strict(X[c], col_name=c)
                vals = vals.where(vals >= 0, np.nan)
                X[c] = np.log1p(vals)
        return X


class NumericWinsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, cols: List[str], lower_q: float = 0.01, upper_q: float = 0.99):
        self.cols = cols
        self.lower_q = lower_q
        self.upper_q = upper_q
        self.bounds_ = {}

    def fit(self, X, y=None):
        for col in self.cols:
            if col in X.columns:
                s = _to_numeric_strict(X[col], col_name=col)
                lo = s.quantile(self.lower_q)
                hi = s.quantile(self.upper_q)
                if pd.notna(lo) and pd.notna(hi):
                    self.bounds_[col] = (lo, hi)
        return self

    def transform(self, X):
        X = X.copy()
        for col, (lo, hi) in self.bounds_.items():
            if col in X.columns:
                X[col] = _to_numeric_strict(X[col], col_name=col).clip(lower=lo, upper=hi)
        return X


class MissingIndicatorAndMedianImputer(BaseEstimator, TransformerMixin):
    def __init__(self, numeric_cols: List[str]):
        self.numeric_cols = numeric_cols
        self.medians_ = {}

    def fit(self, X, y=None):
        for col in self.numeric_cols:
            if col in X.columns:
                self.medians_[col] = _to_numeric_strict(X[col], col_name=col).median()
        return self

    def transform(self, X):
        X = X.copy()
        for col, med in self.medians_.items():
            if col in X.columns:
                vals = _to_numeric_strict(X[col], col_name=col)
                X[col + "_missing"] = vals.isna().astype(int)
                X[col] = vals.fillna(med)
        return X

In [5]:
# Build cleaning pipeline with the defined transformers. This can be fit on the training data and then applied to train/val/test splits separately to avoid data leakage.
def build_cleaning_pipeline(
    numeric_cols: List[str],
    categorical_cols: List[str],
    log_cols: Optional[List[str]] = None
):
    derived_num_cols = [
        "revol_to_loan",
        "installment_to_monthly_income",
        "open_to_total_ratio",
        "fico_times_revolutil",
        "revol_util_gt_100"
    ]
    all_num_cols = list(dict.fromkeys(numeric_cols + derived_num_cols))

    steps = [
        ("parse_dates", ParseDatesTransformer(date_cols=["issue_d", "earliest_cr_line"])),
        ("date_features", DateFeaturesTransformer(date_col="issue_d")),
        ("percent_to_float", PercentToFloatTransformer(cols=["int_rate", "revol_util"])),
        ("term_extract", TermExtractor(col="term")),
        ("emp_len", EmpLengthTransformer(col="emp_length")),
        ("fico_credit", FicoAndCreditAgeTransformer()),
        ("drop_raw_fico", DropColumnsTransformer(cols=["fico_range_low", "fico_range_high"])),
        ("cat_clean", CategoricalCleaner(cols=categorical_cols, rare_thresh=0.01)),
        ("derived", DerivedFeaturesTransformer()),
    ]

    if log_cols is not None and len(log_cols) > 0:
        steps.append(("log_transform", LogTransformer(cols=log_cols)))

    steps.extend([
        ("winsorize", NumericWinsorizer(cols=all_num_cols, lower_q=0.01, upper_q=0.99)),
        ("impute", MissingIndicatorAndMedianImputer(numeric_cols=all_num_cols)),
        ("drop_non_feature_cols", DropColumnsTransformer(
            cols=["loan_id", "loan_status", "target", "issue_d", "earliest_cr_line", "last_pymnt_d", "horizon_end"]
        )),
    ])

    return Pipeline(steps)

In [6]:
# Split data + prepare fundamental/full frames
max_issue_year = int((SNAPSHOT_DATE - pd.DateOffset(months=HORIZON_MONTHS)).year)
test_year = max_issue_year
val_year = max_issue_year - 1
train_end_year = max_issue_year - 2

df_clean = df_clean.sort_values("issue_d").reset_index(drop=True)

train_df = df_clean[df_clean["issue_d"].dt.year <= train_end_year].copy()
val_df   = df_clean[df_clean["issue_d"].dt.year == val_year].copy()
test_df  = df_clean[df_clean["issue_d"].dt.year == test_year].copy()

if len(train_df) == 0 or len(val_df) == 0 or len(test_df) == 0:
    raise ValueError("One of train/val/test splits is empty. Check horizon or split years.")

print("Split years:", {"train<=": train_end_year, "val": val_year, "test": test_year})
print("Rows total/train/val/test:", len(df_clean), len(train_df), len(val_df), len(test_df))

missing_drop = [c for c in FUNDAMENTAL_DROP if c not in df_clean.columns]
if missing_drop:
    raise KeyError(f"Missing FUNDAMENTAL_DROP columns: {missing_drop}")

train_fund = train_df.drop(columns=FUNDAMENTAL_DROP).copy()
val_fund   = val_df.drop(columns=FUNDAMENTAL_DROP).copy()
test_fund  = test_df.drop(columns=FUNDAMENTAL_DROP).copy()

train_full = train_df.copy()
val_full   = val_df.copy()
test_full  = test_df.copy()

y_train_f = train_fund["target"].astype(int).copy()
y_val_f   = val_fund["target"].astype(int).copy()
y_test_f  = test_fund["target"].astype(int).copy()

y_train_full = train_full["target"].astype(int).copy()
y_val_full   = val_full["target"].astype(int).copy()
y_test_full  = test_full["target"].astype(int).copy()

for d in (train_fund, val_fund, test_fund, train_full, val_full, test_full):
    d.drop(columns=["target"], inplace=True)

Split years: {'train<=': 2015, 'val': 2016, 'test': 2017}
Rows total/train/val/test: 1763639 886770 433890 442979


In [7]:
# Fit cleaning pipelines + transform splits
# IMPORTANT: include issue_d time features in numeric columns
numeric_cols_fund = [
    "annual_inc", "open_acc", "total_acc", "delinq_2yrs", "pub_rec", "dti",
    "revol_bal", "revol_util", "loan_amnt",
    "fico", "credit_age_years", "emp_length_years",
    "issue_d_month", "issue_d_weekday", "issue_d_quarter", "issue_d_year", "issue_d_weekofyear",
    "revol_to_loan", "open_to_total_ratio", "fico_times_revolutil", "revol_util_gt_100"
]
categorical_cols_fund = ["home_ownership", "verification_status", "purpose", "addr_state"]

numeric_cols_full = numeric_cols_fund + ["int_rate", "installment"]
categorical_cols_full = categorical_cols_fund + ["grade", "sub_grade"]

log_cols = ["annual_inc", "revol_bal", "loan_amnt"]

pipeline_logistic = build_cleaning_pipeline(
    numeric_cols=numeric_cols_fund,
    categorical_cols=categorical_cols_fund,
    log_cols=log_cols
)
pipeline_xgb = build_cleaning_pipeline(
    numeric_cols=numeric_cols_fund,
    categorical_cols=categorical_cols_fund,
    log_cols=None
)
pipeline_xgb_full = build_cleaning_pipeline(
    numeric_cols=numeric_cols_full,
    categorical_cols=categorical_cols_full,
    log_cols=None
)

X_train_fund_clean = pipeline_logistic.fit_transform(train_fund)
X_val_fund_clean   = pipeline_logistic.transform(val_fund)
X_test_fund_clean  = pipeline_logistic.transform(test_fund)

X_train_fund_xgb = pipeline_xgb.fit_transform(train_fund)
X_val_fund_xgb   = pipeline_xgb.transform(val_fund)
X_test_fund_xgb  = pipeline_xgb.transform(test_fund)

X_train_full_xgb = pipeline_xgb_full.fit_transform(train_full)
X_val_full_xgb   = pipeline_xgb_full.transform(val_full)
X_test_full_xgb  = pipeline_xgb_full.transform(test_full)

assert X_train_fund_clean.shape[0] == len(y_train_f)
assert X_val_fund_clean.shape[0] == len(y_val_f)
assert X_test_fund_clean.shape[0] == len(y_test_f)

print("revol_util_gt_100 in cleaned data:", "revol_util_gt_100" in X_train_fund_clean.columns)
print("issue_d_year in cleaned data:", "issue_d_year" in X_train_fund_clean.columns)
print("Shapes:", X_train_fund_clean.shape, X_train_fund_xgb.shape, X_train_full_xgb.shape)

/Users/alex._choo/anaconda3/envs/am126/lib/python3.13/site-packages/sklearn/pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/alex._choo/anaconda3/envs/am126/lib/python3.13/site-packages/sklearn/pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/alex._choo/anaconda3/envs/am126/lib/python3.13/site-packages/sklearn/pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(
/Users/alex._cho

revol_util_gt_100 in cleaned data: True
issue_d_year in cleaned data: True
Shapes: (886770, 49) (886770, 49) (886770, 57)


/Users/alex._choo/anaconda3/envs/am126/lib/python3.13/site-packages/sklearn/pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


In [8]:
# Logistic
display(X_train_fund_clean.head())

# XGB (fundamental)
display(X_train_fund_xgb.head())

# XGB (full)
display(X_train_full_xgb.head())

,loan_amnt,term,emp_length,home_ownership,annual_inc,verification_status,purpose,addr_state,dti,delinq_2yrs,...,emp_length_years_missing,issue_d_month_missing,issue_d_weekday_missing,issue_d_quarter_missing,issue_d_year_missing,issue_d_weekofyear_missing,revol_to_loan_missing,open_to_total_ratio_missing,fico_times_revolutil_missing,revol_util_gt_100_missing
0,9.259226,36.0,3 years,rent,11.002117,not verified,other,ct,19.50,0.0,...,0,0,0,0,0,0,0,1,1,1
1,8.517393,36.0,10+ years,mortgage,11.156265,not verified,other,ct,8.81,0.0,...,0,0,0,0,0,0,0,1,1,1
2,8.922792,36.0,< 1 year,own,9.998843,not verified,debt_consolidation,ma,14.29,1.0,...,0,0,0,0,0,0,0,0,0,0
3,7.834392,36.0,< 1 year,rent,11.608245,not verified,other,ny,10.00,0.0,...,0,0,0,0,0,0,0,1,1,1
4,7.496097,36.0,< 1 year,rent,9.852247,not verified,other,ma,10.00,0.0,...,0,0,0,0,0,0,0,1,1,1


,loan_amnt,term,emp_length,home_ownership,annual_inc,verification_status,purpose,addr_state,dti,delinq_2yrs,...,emp_length_years_missing,issue_d_month_missing,issue_d_weekday_missing,issue_d_quarter_missing,issue_d_year_missing,issue_d_weekofyear_missing,revol_to_loan_missing,open_to_total_ratio_missing,fico_times_revolutil_missing,revol_util_gt_100_missing
0,10500.0,36.0,3 years,rent,60000.0,not verified,other,ct,19.50,0.0,...,0,0,0,0,0,0,0,1,1,1
1,5000.0,36.0,10+ years,mortgage,70000.0,not verified,other,ct,8.81,0.0,...,0,0,0,0,0,0,0,1,1,1
2,7500.0,36.0,< 1 year,own,22000.0,not verified,debt_consolidation,ma,14.29,1.0,...,0,0,0,0,0,0,0,0,0,0
3,2525.0,36.0,< 1 year,rent,110000.0,not verified,other,ny,10.00,0.0,...,0,0,0,0,0,0,0,1,1,1
4,1800.0,36.0,< 1 year,rent,19000.0,not verified,other,ma,10.00,0.0,...,0,0,0,0,0,0,0,1,1,1


,loan_amnt,term,int_rate,installment,grade,sub_grade,emp_length,home_ownership,annual_inc,verification_status,...,issue_d_quarter_missing,issue_d_year_missing,issue_d_weekofyear_missing,revol_to_loan_missing,open_to_total_ratio_missing,fico_times_revolutil_missing,revol_util_gt_100_missing,int_rate_missing,installment_missing,installment_to_monthly_income_missing
0,10500.0,36.0,11.22,344.87,c,c4,3 years,rent,60000.0,not verified,...,0,0,0,0,1,1,1,0,0,0
1,5000.0,36.0,7.75,156.11,a,a3,10+ years,mortgage,70000.0,not verified,...,0,0,0,0,1,1,1,0,0,0
2,7500.0,36.0,13.75,255.43,e,e2,< 1 year,own,22000.0,not verified,...,0,0,0,0,0,0,0,0,0,0
3,2525.0,36.0,9.33,80.69,b,b3,< 1 year,rent,110000.0,not verified,...,0,0,0,0,1,1,1,0,0,0
4,1800.0,36.0,9.64,60.66,b,b4,< 1 year,rent,19000.0,not verified,...,0,0,0,0,1,1,1,0,0,0


In [9]:
display(X_train_fund_clean.head())
display(X_train_fund_clean.describe().T)
print(X_train_fund_clean.shape)
print(X_train_fund_clean.dtypes)
print(X_train_fund_clean.isna().sum().sort_values(ascending=False).head(10))

,loan_amnt,term,emp_length,home_ownership,annual_inc,verification_status,purpose,addr_state,dti,delinq_2yrs,...,emp_length_years_missing,issue_d_month_missing,issue_d_weekday_missing,issue_d_quarter_missing,issue_d_year_missing,issue_d_weekofyear_missing,revol_to_loan_missing,open_to_total_ratio_missing,fico_times_revolutil_missing,revol_util_gt_100_missing
0,9.259226,36.0,3 years,rent,11.002117,not verified,other,ct,19.50,0.0,...,0,0,0,0,0,0,0,1,1,1
1,8.517393,36.0,10+ years,mortgage,11.156265,not verified,other,ct,8.81,0.0,...,0,0,0,0,0,0,0,1,1,1
2,8.922792,36.0,< 1 year,own,9.998843,not verified,debt_consolidation,ma,14.29,1.0,...,0,0,0,0,0,0,0,0,0,0
3,7.834392,36.0,< 1 year,rent,11.608245,not verified,other,ny,10.00,0.0,...,0,0,0,0,0,0,0,1,1,1
4,7.496097,36.0,< 1 year,rent,9.852247,not verified,other,ma,10.00,0.0,...,0,0,0,0,0,0,0,1,1,1


,count,mean,std,min,25%,50%,75%,max
loan_amnt,886770.0,9.412199,0.655993,7.496097,8.987322,9.472782,9.903538,10.463132
term,886770.0,43.201597,10.998885,36.000000,36.000000,36.000000,60.000000,60.000000
annual_inc,886770.0,11.074404,0.509625,9.852247,10.714440,11.082158,11.407576,12.429220
dti,886770.0,18.128409,8.254703,1.980000,11.910000,17.660000,23.950000,37.480000
delinq_2yrs,886770.0,0.296141,0.722724,0.000000,0.000000,0.000000,0.000000,4.000000
open_acc,886770.0,11.504190,5.109946,3.000000,8.000000,11.000000,14.000000,28.000000
pub_rec,886770.0,0.178407,0.444214,0.000000,0.000000,0.000000,0.000000,2.000000
revol_bal,886770.0,9.300287,0.971100,5.739793,8.771370,9.382443,9.944342,11.444907
revol_util,886770.0,55.063280,23.723076,2.200000,37.700000,56.000000,73.600000,98.500000
total_acc,886770.0,25.211959,11.547676,6.000000,17.000000,24.000000,32.000000,60.000000


(886770, 49)
loan_amnt                              float64
term                                   float64
emp_length                              object
home_ownership                  string[python]
annual_inc                             float64
verification_status             string[python]
purpose                         string[python]
addr_state                      string[python]
dti                                    float64
delinq_2yrs                            float64
open_acc                               float64
pub_rec                                float64
revol_bal                              float64
revol_util                             float64
total_acc                              float64
issue_d_month                            int32
issue_d_weekday                          int32
issue_d_quarter                          int32
issue_d_year                             int32
issue_d_weekofyear                     float64
emp_length_years                       float64


In [10]:
# Save cleaning artifacts (for data_modeling.ipynb)
ARTIFACT_DIR = Path("artifacts/cleaning_only")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# 1) Save fitted cleaning pipelines
cleaning_pipelines = {
    "pipeline_logistic": pipeline_logistic,
    "pipeline_xgb_fund": pipeline_xgb,
    "pipeline_xgb_full": pipeline_xgb_full,
}
joblib.dump(cleaning_pipelines, ARTIFACT_DIR / "cleaning_pipelines.joblib", compress=3)

# 2) Save cleaned outputs + labels
cleaned_outputs = {
    "X_train_fund_clean": X_train_fund_clean,
    "X_val_fund_clean": X_val_fund_clean,
    "X_test_fund_clean": X_test_fund_clean,
    "X_train_fund_xgb": X_train_fund_xgb,
    "X_val_fund_xgb": X_val_fund_xgb,
    "X_test_fund_xgb": X_test_fund_xgb,
    "X_train_full_xgb": X_train_full_xgb,
    "X_val_full_xgb": X_val_full_xgb,
    "X_test_full_xgb": X_test_full_xgb,
    "y_train_f": y_train_f.to_numpy(dtype=np.int8),
    "y_val_f": y_val_f.to_numpy(dtype=np.int8),
    "y_test_f": y_test_f.to_numpy(dtype=np.int8),
    "y_train_full": y_train_full.to_numpy(dtype=np.int8),
    "y_val_full": y_val_full.to_numpy(dtype=np.int8),
    "y_test_full": y_test_full.to_numpy(dtype=np.int8),
}
joblib.dump(cleaned_outputs, ARTIFACT_DIR / "cleaned_outputs.joblib", compress=3)

# 3) Save metadata
meta = {
    "snapshot_date": str(SNAPSHOT_DATE.date()),
    "horizon_months": HORIZON_MONTHS,
    "split_years": {"train_end_year": train_end_year, "val_year": val_year, "test_year": test_year},
    "fundamental_drop": FUNDAMENTAL_DROP,
    "non_default_final_statuses": sorted(list(NON_DEFAULT_FINAL_STATUSES)),
    "active_nondefault_statuses": sorted(list(ACTIVE_NONDEFAULT_STATUSES)),
    "default_statuses": sorted(list(DEFAULT_STATUSES)),
    "target_definition_note": (
        "Default timing is approximated using last_pymnt_d because explicit default event timestamps "
        "are not included in the selected raw columns."
    ),
}
joblib.dump(meta, ARTIFACT_DIR / "meta.joblib", compress=3)

print("Saved artifacts:")
print(" -", ARTIFACT_DIR / "cleaning_pipelines.joblib")
print(" -", ARTIFACT_DIR / "cleaned_outputs.joblib")
print(" -", ARTIFACT_DIR / "meta.joblib")

Saved artifacts:
 - artifacts/cleaning_only/cleaning_pipelines.joblib
 - artifacts/cleaning_only/cleaned_outputs.joblib
 - artifacts/cleaning_only/meta.joblib
